In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

Set plotting style for EDA exports

In [ ]:
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

==========================================<br>
1. DATA LOADING & MERGING<br>
==========================================

In [ ]:
print("Loading datasets...")
client_df = pd.read_csv('telecom/Client.csv')    # [cite: 111]
record_df = pd.read_csv('telecom/Record.csv')    # [cite: 112]

Merge data tightly on Customer_ID [cite: 113]

In [ ]:
df = pd.merge(client_df, record_df, on='Customer_ID')
print(f"Successfully merged. Full dataset shape: {df.shape}\n")

==========================================<br>
2. EXPLORATORY DATA ANALYSIS (EDA) GENERATION<br>
==========================================

In [ ]:
print("Generating EDA insights for your presentation slides...")

Insight A: Churn Rate Baseline

In [ ]:
churn_rate = df['churn'].value_counts(normalize=True) * 100
print(f"--- Baseline Churn Distribution ---")
print(f"Retained (0): {churn_rate[0]:.2f}% | Churned (1): {churn_rate[1]:.2f}%\n")

Insight B: Equipment Age vs Churn

In [ ]:
plt.figure()
sns.boxplot(data=df, x='churn', y='eqpdays', palette='Set2')
plt.title('Handset Equipment Age (Days) vs. Customer Churn')
plt.xlabel('Churn (0 = Kept, 1 = Left)')
plt.ylabel('Days of Current Equipment')
plt.savefig('eda_equipment_age_vs_churn.png', dpi=300)
plt.close()
print("Saved: eda_equipment_age_vs_churn.png")

Insight C: Usage Drops vs Churn

In [ ]:
plt.figure()
sns.kdeplot(data=df, x='change_mou', hue='churn', common_norm=False, fill=True, alpha=0.5)
plt.xlim(-500, 500) # Limiting x-axis to filter extreme outliers for visual clarity [cite: 104]
plt.title('Percentage Change in Minutes of Use (MOU) vs. Churn')
plt.xlabel('% Change in Monthly MOU')
plt.savefig('eda_usage_change_vs_churn.png', dpi=300)
plt.close()
print("Saved: eda_usage_change_vs_churn.png\n")

==========================================<br>
3. FEATURE ENGINEERING & PREPROCESSING<br>
==========================================<br>
Define target and raw features [cite: 62, 79]

In [ ]:
y = df['churn']
X = df.drop(columns=['churn', 'Customer_ID'])

Identify feature types automatically

In [ ]:
numeric_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X.select_dtypes(include=['object']).columns.tolist()

Define structural pipelines for robust handling of missing fields 

In [ ]:
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')), # Safe against extreme outliers [cite: 104]
    ('scaler', StandardScaler())
])

In [ ]:
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')), # Handle missing category tags
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

==========================================<br>
4. TRAIN-TEST SPLIT<br>
==========================================

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

==========================================<br>
5. MODELING & EXPERIMENTATION<br>
==========================================<br>
Model 1: Baseline Logistic Regression [cite: 65, 84]

In [ ]:
lr_pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                               ('classifier', LogisticRegression(max_iter=1000, random_state=42))])

Model 2: Advanced Random Forest Classifier 

In [ ]:
rf_pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                               ('classifier', RandomForestClassifier(n_estimators=100, max_depth=12, random_state=42, n_jobs=-1))])

Execute training [cite: 85]

In [ ]:
print("Training Baseline Model (Logistic Regression)...")
lr_pipeline.fit(X_train, y_train)

In [ ]:
print("Training Target Model (Random Forest)...")
rf_pipeline.fit(X_train, y_train)
print("Training complete.\n")

==========================================<br>
6. EVALUATION METRICS REPORTING<br>
==========================================

In [ ]:
def evaluate_model(pipeline, name, X_test, y_test):
    predictions = pipeline.predict(X_test)
    probabilities = pipeline.predict_proba(X_test)[:, 1]
    
    auc_score = roc_auc_score(y_test, probabilities)
    cm = confusion_matrix(y_test, predictions)
    report = classification_report(y_test, predictions, output_dict=True)
    
    # Extract metrics targeted for slides 
    print(f"==========================================")
    print(f"SLIDE REQUIREMENT DATA FOR: {name}")
    print(f"==========================================")
    print(f"Model Name: {name}")
    print(f"Primary Evaluation Metric: ROC-AUC")
    print(f"Score (ROC-AUC): {auc_score:.4f}")
    print(f"Recall (Class 1 - Churn Capture Rate): {report['1']['recall']:.4f}")
    print(f"Precision (Class 1): {report['1']['precision']:.4f}")
    print(f"\nConfusion Matrix:\n{cm}")
    print(f"==========================================\n")
    return probabilities

In [ ]:
rf_probs = evaluate_model(lr_pipeline, "Logistic Regression Baseline", X_test, y_test)
rf_probs = evaluate_model(rf_pipeline, "Random Forest Classifier", X_test, y_test)

==========================================<br>
7. QUANTIFIED BUSINESS IMPACT CALCULATOR<br>
==========================================<br>
Let's translate the ML output to executive numbers [cite: 23, 90]

In [ ]:
print("--- Quantifying Business Impact for Executives ---")

Assumptions based on common industry variables [cite: 92]

In [ ]:
customer_base_tested = len(y_test)
estimated_clv = 4000  # Customer Lifetime Value in INR
retention_offer_cost = 800  # Cost of promotional upgrade incentive per customer
acceptance_rate = 0.40  # 40% of targeted churning customers accept the save offer

Let's target top 10% highest risk customers according to our Random Forest model

In [ ]:
test_results = pd.DataFrame({'true_churn': y_test, 'risk_score': rf_probs})
top_10_percent_cutoff = test_results['risk_score'].quantile(0.90)
high_risk_campaign = test_results[test_results['risk_score'] >= top_10_percent_cutoff]

In [ ]:
actual_churners_targeted = high_risk_campaign['true_churn'].sum()
total_customers_targeted = len(high_risk_campaign)

Calculate financial impact calculations [cite: 91]

In [ ]:
churners_saved = int(actual_churners_targeted * acceptance_rate)
revenue_protected = churners_saved * estimated_clv
campaign_expenditure = total_customers_targeted * retention_offer_cost
net_savings = revenue_protected - campaign_expenditure

In [ ]:
print(f"Out of {customer_base_tested} evaluated subscribers, the top 10% high-risk cohort captures {actual_churners_targeted} true churners.")
print(f"With a proactive retention strategy offering device upgrades[cite: 23, 93]:")
print(f" * Estimated Churning Customers Saved: {churners_saved}")
print(f" * Revenue Protected: INR {revenue_protected:,}")
print(f" * Total Campaign Cost: INR {campaign_expenditure:,}")
print(f" * **Net Quantified Cost Savings**: INR {net_savings:,} [cite: 94]")
print(f"==========================================")